# HADDOCK3 GA1 ligand-pose clustering -- good poses only, importer + non-importer

**Kernel:** `abcfold-npf-notebook` (`envs/notebook.yaml`)

For each `NPF_LDA_kernel` GA1 protein (importer AND non-importer -- see
below for why both are in scope now), pools **every kept `[flexref]`
model** across all 3 of that protein's `ca_cluster` redocked complexes (up
to 40 models/complex x 3 complexes = up to 120 poses/protein -- not just
the single top-ranked model `compare_to_abcfold.py`/
`rescore_redocked_batch.py` use), and asks: does HADDOCK3's own docking
swarm actually converge on one binding mode, or several distinct ones --
and do the better-scoring poses cluster together?

**2026-08-26 update -- bad/off-target poses are now filtered out first.**
The first version of this notebook (importer-only) pooled EVERY kept pose
with no filtering, and the user noticed by eye that a real fraction of the
resulting PCA clusters sat where the membrane should be, not anywhere near
the CDD pocket -- pure noise from HADDOCK3 occasionally finding a
nonspecific, non-pocket pose that still scores deceptively well.
Confirmed quantitatively in `redocking/src/pose_pocket_engagement.py`
(Stage 8, run against every kept model of every complex, both roles): a
GMM(2) fit on CDD active-residue contact count is sharply bimodal
(component means 0.52 vs 11.43 contacts, weights 0.43/0.57) -- so **every
pose in this notebook is now filtered to the "good" (pocket-engaging) GMM
component before alignment/PCA/clustering even starts** (see
`results/comparison/pose_pocket_engagement.csv`). Non-importers are now
in scope too, since with the noise filtered out this is finally a fair
question to ask of them as well: does a non-importer protein's dominant
docked binding mode also sit in/near its own CDD-annotated pocket, or
somewhere else entirely?

**Method** (mirrors `scripts/cluster_conformations.py`'s own ligand-pose
clustering: Kabsch-fit re-alignment -> all-heavy-atom ligand PCA -> GMM
auto (BIC-knee) -- reusing that script's own `_kabsch_fit`/
`_fit_gmm_bic_sweep`/`GMM_PALETTE` helpers directly, not reimplementing
them):

1. **Filter to `good_pose` only** (see above) before anything else.
2. **C-alpha alignment (needed even within one complex, not just across
   the 3 `ca_cluster`s)** -- `[flexref]` is a semi-flexible refinement, so
   even the kept models from the SAME complex don't share one exact
   backbone frame, and the 3 `ca_cluster`s are three genuinely different
   macro-conformations to begin with. Every model's receptor chain "A"
   C-alpha coordinates (shared-residue intersection across all of that
   protein's models) are Kabsch-superposed onto one reference model
   (that protein's lowest-numbered `ca_cluster` that has any good poses at
   all, best HADDOCK score among those -- some individual `ca_cluster`s
   have ZERO good poses, see Stage 8's findings, so `ca_cluster 0` isn't
   always available as the reference) before the ligand is looked at at
   all -- otherwise "where the ligand sits" would partly just be measuring
   "how the backbone moved."
3. **PCA** on the aligned ligand's all-heavy-atom xyz (flattened, GA1's own
   fixed positional atom order -- see `ligand_fix.py`), 2 components.
4. **GMM, auto k (BIC-knee sweep)** on the 2-D embedding -- same method
   `scripts/cluster_conformations.py` already uses for ABCfold's own
   ligand-pose clustering, imported directly rather than re-derived.
5. **Symlinks**: one `results/ligand_pose_clusters/<protein>/cluster_<k>/`
   per cluster, with the top-`max_per_cluster`-by-HADDOCK-score members of
   that cluster symlinked in (pointing at the real, already-gzipped
   `4_flexref/flexref_<n>.pdb.gz` under `redocking/results/haddock_runs/`
   -- nothing is copied), plus one `assignments.csv` per protein recording
   every (surviving, good) pose's `(pc1, pc2, cluster, haddock_score,
   n_active_residues_contacted, symlinked)`.
6. **Plots**: PCA scatter colored continuously by HADDOCK score (not by
   cluster id) with each GMM cluster's covariance ellipse drawn as an
   overlay -- so cluster membership and score both read off the same plot
   at once. Also prints, per protein, which cluster is most populated and
   its mean CDD-contact count vs. the protein's overall mean -- a direct,
   numeric answer to "is the dominant binding mode also the one actually
   engaging the real pocket" instead of an eyeballed one.

Standalone exploratory notebook -- run after `redocking/`'s HADDOCK3 array
(Stage 6) AND `pose_pocket_engagement.py` (Stage 8) have both completed.


In [ ]:
import sys
from pathlib import Path

import gemmi
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.decomposition import PCA

ROOT = Path("..")
REDOCKING_ROOT = ROOT / "redocking"
HADDOCK_RUNS_DIR = REDOCKING_ROOT / "results" / "haddock_runs"
MANIFEST_CSV = REDOCKING_ROOT / "data" / "manifest.csv"
POCKET_ENGAGEMENT_CSV = REDOCKING_ROOT / "results" / "comparison" / "pose_pocket_engagement.csv"
OUT_ROOT = REDOCKING_ROOT / "results" / "ligand_pose_clusters"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Reuse the pipeline's own Kabsch-fit + GMM-auto-BIC-knee helpers directly
# (see markdown above) rather than reimplementing them.
sys.path.insert(0, str(ROOT / "scripts"))
from cluster_conformations import _kabsch_fit, _fit_gmm_bic_sweep, GMM_PALETTE  # noqa: E402

RECEPTOR_CHAIN = "A"
LIGAND_CHAIN_HADDOCK = "B"
MAX_PER_CLUSTER = 10  # symlink cap per cluster, same default cluster_conformations.py uses

# Stage 8's per-(complex_id, caprieval_rank) good/bad classification --
# see the markdown above for why every pose gets filtered against this
# before alignment/PCA/clustering even starts.
pocket_engagement = pd.read_csv(POCKET_ENGAGEMENT_CSV)
GOOD_POSE_LOOKUP = {
    (row.complex_id, row.caprieval_rank): row.good_pose for row in pocket_engagement.itertuples()
}
CONTACTS_LOOKUP = {
    (row.complex_id, row.caprieval_rank): row.n_active_residues_contacted for row in pocket_engagement.itertuples()
}


## Load every protein's full flexref ensemble, good poses only

One row per (complex_id, flexref model) that Stage 8 classified
`good_pose`: receptor C-alpha coordinates (as a `{resnr: xyz}` dict, for
the shared-residue Kabsch fit below), ligand heavy-atom coordinates (fixed
positional order), that model's own HADDOCK score straight from
`capri_ss.tsv`, and its CDD active-residue contact count (from
`pose_pocket_engagement.csv`, reused rather than recomputed). `_model_path`'s
`.gz` fallback mirrors `compare_to_abcfold.py`/`rescore_redocked_batch.py`'s
own fix (HADDOCK3 gzips every kept model in place after writing
`capri_ss.tsv` -- see `reference_haddock3_cns_ligand_param_propagation`
memory point 8).


In [ ]:
import csv


def _model_path(caprieval_dir: Path, row: dict) -> Path:
    model_path = Path(row["model"])
    if not model_path.is_absolute():
        model_path = caprieval_dir / model_path
    if not model_path.exists():
        gz_path = model_path.with_suffix(model_path.suffix + ".gz")
        if gz_path.exists():
            return gz_path
    return model_path


def _final_caprieval_dir(run_dir: Path) -> Path:
    candidates = sorted(run_dir.glob("*_caprieval"), key=lambda p: int(p.name.split("_")[0]))
    return candidates[-1]


def all_complexes() -> pd.DataFrame:
    return pd.read_csv(MANIFEST_CSV).sort_values(["protein", "ca_cluster"])


def load_flexref_ensemble(protein: str) -> list[dict]:
    """Every kept flexref model that Stage 8 classified `good_pose`, across
    all of this protein's ca_cluster complexes -- bad/off-target poses are
    dropped here, before alignment/PCA ever sees them (see markdown
    above)."""
    complexes = all_complexes()
    complexes = complexes[complexes["protein"] == protein]
    records = []
    n_dropped = 0
    for _, row in complexes.iterrows():
        complex_id, ca_cluster = row["complex_id"], int(row["ca_cluster"])
        run_dir = HADDOCK_RUNS_DIR / complex_id
        caprieval_dir = _final_caprieval_dir(run_dir)
        with (caprieval_dir / "capri_ss.tsv").open() as f:
            capri_rows = list(csv.DictReader(f, delimiter="\t"))
        for capri_row in capri_rows:
            rank = int(capri_row["caprieval_rank"])
            if not GOOD_POSE_LOOKUP.get((complex_id, rank), False):
                n_dropped += 1
                continue
            model_path = _model_path(caprieval_dir, capri_row)
            st = gemmi.read_structure(str(model_path))
            st.setup_entities()
            ca = {}
            ligand_atoms = []
            for chain in st[0]:
                if chain.name == RECEPTOR_CHAIN:
                    for res in chain:
                        atom = res.find_atom("CA", "\0")
                        if atom is not None:
                            ca[res.seqid.num] = np.array([atom.pos.x, atom.pos.y, atom.pos.z])
                elif chain.name == LIGAND_CHAIN_HADDOCK:
                    for res in chain:
                        for atom in res:
                            if atom.element.name != "H":
                                ligand_atoms.append([atom.pos.x, atom.pos.y, atom.pos.z])
            records.append({
                "complex_id": complex_id, "protein": protein, "role": row["role"], "ca_cluster": ca_cluster,
                "caprieval_rank": rank, "haddock_score": float(capri_row["score"]),
                "n_active_residues_contacted": CONTACTS_LOOKUP[(complex_id, rank)],
                "model_path": model_path,
                "ca": ca, "ligand_coords": np.array(ligand_atoms),
            })
    print(f"{protein}: {len(records)} good poses kept, {n_dropped} bad/off-target poses filtered out")
    return records


## Align (Kabsch, shared C-alpha residues) and PCA-embed

The reference frame for a given protein is the best-HADDOCK-score good
pose from that protein's LOWEST-numbered `ca_cluster` that has any good
poses at all -- not always `ca_cluster 0`/rank 1 (some individual
`ca_cluster`s have zero good poses after Stage 8's filtering). Every other
good pose gets Kabsch-superposed onto it using whichever C-alpha residue
numbers are present in BOTH (receptors from different `ca_cluster`s can
have slightly different resolved ranges), then that same rotation/
translation is applied to the ligand's own heavy atoms before PCA.


In [ ]:
def _pick_reference(records: list[dict]) -> dict:
    min_cluster = min(r["ca_cluster"] for r in records)
    candidates = [r for r in records if r["ca_cluster"] == min_cluster]
    return min(candidates, key=lambda r: r["haddock_score"])


def align_and_featurize(records: list[dict]) -> np.ndarray:
    ref = _pick_reference(records)
    ref_ca = ref["ca"]

    aligned_ligands = []
    for rec in records:
        shared = sorted(set(rec["ca"]) & set(ref_ca))
        if len(shared) < 10:
            raise ValueError(f"{rec['complex_id']} rank {rec['caprieval_rank']}: only {len(shared)} "
                              "shared C-alpha residues with the reference -- numbering mismatch?")
        mobile = np.array([rec["ca"][i] for i in shared])
        target = np.array([ref_ca[i] for i in shared])
        R, t = _kabsch_fit(mobile, target)
        aligned = rec["ligand_coords"] @ R.T + t
        aligned_ligands.append(aligned.flatten())
    return np.array(aligned_ligands)


def cluster_protein(protein: str) -> tuple[pd.DataFrame, dict]:
    """Returns (assignments df, {cluster_id: (mean, cov)} for the ellipse
    overlay) for one protein's good-pose flexref ensemble."""
    records = load_flexref_ensemble(protein)
    X = align_and_featurize(records)

    pca = PCA(n_components=2, random_state=42)
    xy = pca.fit_transform(X)

    gmm, k, bic_by_k = _fit_gmm_bic_sweep(xy, k_min=1, k_max=min(8, len(records) - 1))
    labels = gmm.predict(xy)

    df = pd.DataFrame({
        "complex_id": [r["complex_id"] for r in records],
        "role": [r["role"] for r in records],
        "ca_cluster": [r["ca_cluster"] for r in records],
        "caprieval_rank": [r["caprieval_rank"] for r in records],
        "haddock_score": [r["haddock_score"] for r in records],
        "n_active_residues_contacted": [r["n_active_residues_contacted"] for r in records],
        "model_path": [r["model_path"] for r in records],
        "pc1": xy[:, 0], "pc2": xy[:, 1],
        "cluster": labels,
    })
    ellipses = {c: (gmm.means_[c], gmm.covariances_[c]) for c in range(k)}
    evr = pca.explained_variance_ratio_
    print(f"{protein} ({df['role'].iloc[0]}): {len(records)} good poses, "
          f"PCA explained variance PC1={evr[0]:.1%} PC2={evr[1]:.1%}, GMM auto k={k}")

    sizes = df["cluster"].value_counts()
    top_cluster = sizes.idxmax()
    overall_mean = df["n_active_residues_contacted"].mean()
    top_mean = df.loc[df["cluster"] == top_cluster, "n_active_residues_contacted"].mean()
    print(f"  most-populated cluster = {top_cluster} ({sizes[top_cluster]}/{len(df)} poses, "
          f"{sizes[top_cluster] / len(df):.1%}) -- mean CDD contacts {top_mean:.1f} "
          f"vs. protein overall mean {overall_mean:.1f}")

    return df, ellipses


## Symlink cluster representatives

Per cluster, the top `MAX_PER_CLUSTER` poses by HADDOCK score (most
negative/best first) get a real symlink written into
`results/ligand_pose_clusters/<protein>/cluster_<k>/` -- pointing straight
at the real (already-gzipped) file under `results/haddock_runs/`, nothing
copied. `assignments["symlinked"]` marks which rows actually got a link
(every row is kept in the table regardless, same convention
`cluster_conformations.py`'s own `assignments.parquet` uses).


In [ ]:
def symlink_clusters(protein: str, df: pd.DataFrame, max_per_cluster: int = MAX_PER_CLUSTER) -> pd.DataFrame:
    protein_dir = OUT_ROOT / protein
    if protein_dir.exists():
        for stale in protein_dir.glob("cluster_*/*"):
            if stale.is_symlink():
                stale.unlink()

    df = df.sort_values(["cluster", "haddock_score"]).copy()
    df["symlinked"] = False
    n_symlinked = 0
    for cluster_id, group in df.groupby("cluster"):
        cluster_dir = protein_dir / f"cluster_{cluster_id}"
        cluster_dir.mkdir(parents=True, exist_ok=True)
        for idx, row in group.head(max_per_cluster).iterrows():
            dest = cluster_dir / f"{row.complex_id}__rank{row.caprieval_rank}.pdb.gz"
            if not dest.exists():
                dest.symlink_to(Path(row.model_path).resolve())
            df.loc[idx, "symlinked"] = True
            n_symlinked += 1

    out_csv = OUT_ROOT / f"{protein}_assignments.csv"
    df.drop(columns=["model_path"]).assign(model_path=df["model_path"].astype(str)).to_csv(out_csv, index=False)
    print(f"{protein}: {n_symlinked}/{len(df)} poses symlinked across {df['cluster'].nunique()} clusters -> {out_csv}")
    return df


## Plot: PCA embedding, colored by HADDOCK score, GMM clusters as ellipses

Marker color = HADDOCK score (continuous, `RdBu_r` reversed so the
best/most-negative scores read as the "hot" end); one dashed ellipse per
GMM cluster (1.5 std, `scripts/cluster_conformations.py`'s own convention)
so cluster membership and score are both visible on the one plot.


In [ ]:
def _ellipse_xy(mean, cov, n_std=1.5, n_pts=80):
    vals, vecs = np.linalg.eigh(cov)
    vals = np.clip(vals, 0, None)
    theta = np.linspace(0, 2 * np.pi, n_pts)
    circle = np.stack([np.cos(theta), np.sin(theta)])
    ellipse = vecs @ np.diag(n_std * np.sqrt(vals)) @ circle
    return mean[0] + ellipse[0], mean[1] + ellipse[1]


def plot_pose_clusters(protein: str, df: pd.DataFrame, ellipses: dict):
    role = df["role"].iloc[0]
    fig = go.Figure()
    fig.add_scatter(
        x=df["pc1"], y=df["pc2"], mode="markers",
        marker=dict(
            color=df["haddock_score"], colorscale="RdBu_r", reversescale=True,
            colorbar=dict(title="HADDOCK score"), size=9,
            line=dict(color="black", width=0.5),
        ),
        customdata=df[["complex_id", "ca_cluster", "caprieval_rank", "cluster", "n_active_residues_contacted"]],
        hovertemplate=(
            "%{customdata[0]} (ca_cluster %{customdata[1]}, rank %{customdata[2]})<br>"
            "GMM cluster %{customdata[3]}, CDD contacts %{customdata[4]}<br>"
            "score=%{marker.color:.1f}<extra></extra>"
        ),
        name="poses",
    )
    for cluster_id, (mean, cov) in ellipses.items():
        ex, ey = _ellipse_xy(mean, cov)
        color = GMM_PALETTE[cluster_id % len(GMM_PALETTE)]
        fig.add_scatter(x=ex, y=ey, mode="lines", line=dict(color=color, width=2, dash="dash"),
                         name=f"GMM cluster {cluster_id}", showlegend=True)
    fig.update_layout(
        title=f"{protein} ({role}) -- GA1 ligand-pose PCA (good poses only, every ca_cluster pooled)<br>"
              "colored by HADDOCK score, dashed = GMM cluster",
        xaxis_title="PC1", yaxis_title="PC2", width=800, height=650,
    )
    fig.show()


## Interactive: pick one protein (importer or non-importer)


In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output, display

_manifest = all_complexes()
PROTEIN_ROLE = _manifest.drop_duplicates("protein").set_index("protein")["role"].to_dict()
ALL_PROTEINS = sorted(PROTEIN_ROLE)
DROPDOWN_OPTIONS = [(f"{p} ({PROTEIN_ROLE[p]})", p) for p in ALL_PROTEINS]

_out = widgets.Output()
_dropdown = widgets.Dropdown(options=DROPDOWN_OPTIONS, description="protein:")


def _on_change(change):
    with _out:
        clear_output(wait=True)
        protein = change["new"]
        df, ellipses = cluster_protein(protein)
        df = symlink_clusters(protein, df)
        plot_pose_clusters(protein, df, ellipses)
        display(df.drop(columns=["model_path"]).sort_values(["cluster", "haddock_score"]))


_dropdown.observe(_on_change, names="value")
display(_dropdown, _out)
_on_change({"new": ALL_PROTEINS[0]})


## Every protein, one after another

All 24 GA1 redocked proteins -- 5 importer + 19 non-importer -- so the
"most-populated cluster" printout above can be compared across the whole
corpus, not just eyeballed one protein at a time.


In [ ]:
_top_cluster_summary = []
for _protein in ALL_PROTEINS:
    _df, _ellipses = cluster_protein(_protein)
    _df = symlink_clusters(_protein, _df)
    plot_pose_clusters(_protein, _df, _ellipses)
    _sizes = _df["cluster"].value_counts()
    _top = _sizes.idxmax()
    _top_cluster_summary.append({
        "protein": _protein, "role": _df["role"].iloc[0],
        "n_good_poses": len(_df), "n_clusters": _df["cluster"].nunique(),
        "top_cluster_frac": _sizes[_top] / len(_df),
        "top_cluster_mean_contacts": _df.loc[_df["cluster"] == _top, "n_active_residues_contacted"].mean(),
        "overall_mean_contacts": _df["n_active_residues_contacted"].mean(),
    })

pd.DataFrame(_top_cluster_summary).to_csv(OUT_ROOT / "top_cluster_summary.csv", index=False)
